# Phase 2 — B1 Joint Multilingual Training (Kaggle)

Runs B1 on the frozen five-language XNLI protocol. For each selected stream order, it pools all five fixed training subsets, applies the configured seeded shuffle, trains one LoRA adapter for one epoch, evaluates every language, and writes a validated joint result record. Joint training has no sequential performance matrix, so BWT, FWT, and forgetting are intentionally undefined.

## Before running
1. Enable a Kaggle GPU and Internet.
2. Accept the gated model terms for `meta-llama/Llama-3.2-1B` and add a Kaggle secret named `HF_TOKEN`.
3. Add a Kaggle secret named `GH_TOKEN` with read access to this private GitHub repository. For a fine-grained token, grant this repository **Contents: Read-only**. Never paste the token into notebook code.
4. Add optional `WANDB_API_KEY` only if you enable W&B.
5. Run all cells, then use **Save Version** to persist `/kaggle/working/phase2_artifacts` as notebook output. A run interrupted during training leaves its checkpoint; inspect it and use a fresh artifact root to restart that incomplete run.

The full protocol is compute-intensive. Set `ORDER_IDS` to one or more order IDs to run a subset. Completed result files are validated and skipped on rerun.

In [ ]:
from pathlib import Path
import base64
import os
import subprocess
import sys

REPO_URL = "https://github.com/Mani-S-49/asymmetric-interference-continual-learning..git"
REPO_ROOT = Path("/kaggle/working/asymmetric-interference-continual-learning")
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
gh_token = secrets.get_secret("GH_TOKEN")
if not gh_token:
    raise RuntimeError("Add a GH_TOKEN Kaggle secret with read access to the private repository.")
git_env = os.environ.copy()
encoded_auth = base64.b64encode(f"x-access-token:{gh_token}".encode()).decode()
git_env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraheader", "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {encoded_auth}"})
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True, env=git_env)
elif (REPO_ROOT / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_ROOT), "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True, env=git_env)
del gh_token, encoded_auth, git_env
if not (REPO_ROOT / "phase2" / "run_baseline.py").is_file():
    raise FileNotFoundError("The cloned repository is missing the Phase 2 B0/B1 runner. Push the current repository changes, then restart this notebook.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "phase2" / "requirements-phase2.txt")], check=True)
sys.path.insert(0, str(REPO_ROOT))

hf_token = secrets.get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN Kaggle secret is empty.")
os.environ["HF_TOKEN"] = hf_token
del hf_token

USE_WANDB = False
WANDB_PROJECT = "phase2-xnli-baselines"
if USE_WANDB:
    wandb_key = secrets.get_secret("WANDB_API_KEY")
    if not wandb_key:
        raise RuntimeError("Enable USE_WANDB only after adding the WANDB_API_KEY Kaggle secret.")
    os.environ["WANDB_API_KEY"] = wandb_key
    del wandb_key

PROTOCOL_PATH = REPO_ROOT / "phase2" / "configs" / "pilot_protocol.json"
ARTIFACT_ROOT = Path("/kaggle/working/phase2_artifacts")
ORDER_IDS = None  # Example: ["order-1"] or ["order-1", "order-2"]
print({"repo": str(REPO_ROOT), "artifact_root": str(ARTIFACT_ROOT), "wandb": USE_WANDB})

In [ ]:
from phase2.preflight import collect_preflight
from phase2.prepare_data import prepare_xnli
from phase2.protocol import load_protocol
from phase2.run_baseline import validate_prepared_artifacts
import json

preflight = collect_preflight(require_cuda=True, check_gated_model=True)
print(json.dumps(preflight, indent=2, sort_keys=True))

protocol = load_protocol(PROTOCOL_PATH)
if not (ARTIFACT_ROOT / "protocol_manifest.json").is_file():
    if (ARTIFACT_ROOT / "data").exists():
        raise FileExistsError("A partial data directory exists without its manifest. Inspect it before using a fresh artifact root.")
    prepared_manifest = prepare_xnli(PROTOCOL_PATH, ARTIFACT_ROOT)
    print("Prepared fixed XNLI subsets:", prepared_manifest["expected_run_count"], "scheduled runs")
else:
    prepared_manifest = validate_prepared_artifacts(protocol, ARTIFACT_ROOT)
    print("Verified existing fixed data manifest:", prepared_manifest["protocol_sha256"])

print({"languages": protocol["dataset"]["languages"], "orders": [order["id"] for order in protocol["orders"]], "buffer_capacity": protocol["replay"]["buffer_capacity_examples"], "data_manifest_sha256": prepared_manifest["data_manifest_sha256"]})

In [ ]:
import subprocess
import sys

command = [sys.executable, "-m", "phase2.run_baseline", "--method", "B1", "--protocol", str(PROTOCOL_PATH), "--artifact-root", str(ARTIFACT_ROOT)]
for order_id in ORDER_IDS or []:
    command.extend(["--order-id", order_id])
if USE_WANDB:
    command.extend(["--wandb-project", WANDB_PROJECT])
subprocess.run(command, cwd=REPO_ROOT, check=True)

result_paths = sorted((ARTIFACT_ROOT / "results").glob("*-B1-*.json"))
print("B1 result files:", len(result_paths))
for path in result_paths:
    print(path.name)

## Saved outputs

- `phase2_artifacts/protocol_manifest.json` and `data/`: frozen selected examples and checksums.
- `phase2_artifacts/checkpoints/<run_id>/`: trained joint adapter and tokenizer.
- `phase2_artifacts/results/<run_id>.json`: validated joint result record for aggregation.

B1 trains on the same pooled examples for every order permutation. Differences from B0 are intentional: B1 does not perform language-by-language stages and does not report sequential transfer/forgetting metrics.